<a href="https://colab.research.google.com/github/castrokelly/PPGIa/blob/main/LeNet5_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LeNet-5 no MNIST

In [1]:
import os
import cv2
import numpy as np
from matplotlib import pyplot as plt
from google.colab.patches import cv2_imshow

In [2]:
# https://dontpad.com/pucpr-vcap

# função auxiliar para baixar arquivos da internet
import urllib

def download_file(url, filename):
    urllib.request.urlretrieve(url, filename)

In [3]:
# https://dontpad.com/pucpr-vcap

# função auxiliar para mostrar imagens lado a lado
def show_side_by_side(images, titles=None, figsize=None, fontsize=None, wspace=0.05):
    '''Display multiple images in a single row using matplotlib.'''
    n = len(images)

    if figsize:
        plt.figure(figsize=figsize)

    for i in range(n):
        plt.subplot(1, n, i + 1)
        img_rgb = cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB)
        plt.imshow(img_rgb)
        if titles:
            if fontsize:
                plt.title(titles[i], fontsize=fontsize)
            else:
                plt.title(titles[i])
        plt.axis('off')

    # Adjust spacing between subplots
    plt.subplots_adjust(wspace=wspace, hspace=0)
    plt.show()

In [4]:
# https://dontpad.com/pucpr-vcap

import os
import cv2
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score

# reprodutibilidade
# em GPU, ainda podem existir pequenas diferenças entre execuções
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponível?", tf.config.list_physical_devices('GPU') != [])

TensorFlow: 2.20.0
GPU disponível? True


In [5]:
# @title Modelo LeNet-5 "na mão" (tanh + avg-pool, conv válidas de 5x5)
def build_lenet5(input_shape, num_classes):
    """
    Arquitetura LeNet-5 clássica:
    - Conv(6, 5x5, valid) + tanh
    - AveragePool(2x2)
    - Conv(16, 5x5, valid) + tanh
    - AveragePool(2x2)
    - Conv(120, 5x5, valid) + tanh  -> saída 1x1x120 quando entrada for 32x32
    - Flatten
    - Dense(84) + tanh
    - Dense(num_classes) + softmax
    """

    # Camada de entrada, define o formato esperado das imagens
    inputs = keras.Input(shape=input_shape)

    # Primeira convolução:
    # - 6 filtros de tamanho 5x5
    #   cada filtro aprende um padrão diferente
    #   como bordas, transições e estruturas simples
    # - Padding: "valid" (sem padding → não adiciona zeros nas bordas)
    #   por isso, a dimensão diminui de 32x32 para 28x28
    #   pode testar padding='same'
    # - ativação tangente hiperbólica (tanh), como no modelo original
    x = layers.Conv2D(6, kernel_size=5, activation='tanh')(inputs)

    # Primeira camada de pooling:
    # - AveragePooling (ao invés de MaxPooling, seguindo LeNet original)
    #
    # pooling reduz as dimensões espaciais dos mapas de características
    # resumindo pequenas regiões da imagem
    #
    # pool_size=2:
    #   considera uma região 2x2 de cada vez
    #
    # strides=2:
    #   desloca essa janela 2 pixels a cada operação
    #
    # como pool_size=2 e strides=2, as regiões não se sobrepõem
    # e a altura e a largura são reduzidas aproximadamente pela metade
    x = layers.AveragePooling2D(pool_size=2, strides=2)(x)

    # Segunda convolução:
    # - 16 filtros de 5x5
    x = layers.Conv2D(16, kernel_size=5, activation='tanh')(x)

    # Segundo pooling médio, reduzindo novamente pela metade
    x = layers.AveragePooling2D(pool_size=2, strides=2)(x)

    # Terceira convolução:
    # - 120 filtros de 5x5
    x = layers.Conv2D(120, kernel_size=5, activation='tanh')(x)

    # Achata (flatten) para vetor 1D antes das camadas densas
    # No caso de entradas 32x32, resulta em saída 1x1x120
    x = layers.Flatten()(x)

    # Primeira camada totalmente conectada (dense):
    # - 84 neurônios
    #   84 neurônios foram escolhidos empiricamente por LeCun em 1998,
    #   como um compromisso entre poder de representação e custo computacional.
    x = layers.Dense(84, activation='tanh')(x)

    # Camada de saída:
    # - número de neurônios = num_classes
    # - softmax transforma os valores em uma distribuição de probabilidades
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    # Cria o modelo Keras com entrada e saída definidos
    model = keras.Model(inputs, outputs, name="LeNet5")
    return model

In [6]:
# @title Treino/Teste em MNIST (grayscale 28x28 → redimensionar para 32x32)
from tensorflow.keras.datasets import mnist

# 1) Carregar a MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# No Python, os ... (três pontos) dentro de colchetes ou parênteses
# são um atalho do NumPy que significa
# “pegar todos os elementos em todas as dimensões restantes”.

# 2) Preparar: normalizar para [0,1], adicionar canal
# e redimensionar para 32x32
X_train = (X_train / 255.0).astype("float32")[..., None]  # (N, 28, 28, 1)
X_test  = (X_test  / 255.0).astype("float32")[..., None]

# a LeNet-5 original recebe imagens 32x32
X_train = tf.image.resize(X_train, (32, 32)).numpy()
X_test  = tf.image.resize(X_test,  (32, 32)).numpy()

# 3) Separar validação (ex.: 5k amostras do treino)
VAL_SIZE = 5000
X_val, y_val = X_train[:VAL_SIZE], y_train[:VAL_SIZE]
X_train, y_train = X_train[VAL_SIZE:], y_train[VAL_SIZE:]

print("MNIST shapes:",
      "\n  X_tr:", X_train.shape, "y_tr:", y_train.shape,
      "\n  X_val:", X_val.shape, "y_val:", y_val.shape,
      "\n  X_test:", X_test.shape, "y_test:", y_test.shape)

# 4) Construir o modelo
mnist_model = build_lenet5(input_shape=(32, 32, 1), num_classes=10)

# compile define como a rede será otimizada
mnist_model.compile(
    optimizer="rmsprop",
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# A SparseCategoricalCrossentropy é uma função de perda
# usada em classificação multi-classe.
# O “sparse” significa que os rótulos (labels)
# são fornecidos como inteiros (ex.: 0, 1, 2, ..., 9),
# em vez de vetores one-hot.

print()
mnist_model.summary()
print()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
MNIST shapes: 
  X_tr: (55000, 32, 32, 1) y_tr: (55000,) 
  X_val: (5000, 32, 32, 1) y_val: (5000,) 
  X_test: (10000, 32, 32, 1) y_test: (10000,)



Model: "LeNet5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 1, 1, 120)      │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
EPOCHS = 10
BATCH_SIZE = 128

# uma epoch = uma passagem completa pelo conjunto de treinamento
#
# batch size = quantas amostras são processadas
# antes de uma atualização dos pesos

steps_per_epoch = int(np.ceil(len(X_train) / BATCH_SIZE))
print(f'Amostras de treino: {len(X_train):,}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Epochs: {EPOCHS}')
print(f'Atualizações por epoch: aproximadamente {steps_per_epoch:,}')

Amostras de treino: 55,000
Batch size: 128
Epochs: 10
Atualizações por epoch: aproximadamente 430


In [8]:
# durante o fit:
# a rede percorre todos os batches de uma epoch
# e repete esse processo EPOCHS vezes
history_mnist = mnist_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
)

# Avaliação do modelo em dados nunca vistos
# o conjunto de teste só aparece no final
# depois que o treinamento e as decisões de modelagem terminaram
loss, acc = mnist_model.evaluate(X_test, y_test, verbose=0)

print(f"\n[TEST] Acc: {acc*100:.2f}%")

Epoch 1/10
430/430 - 7s - 17ms/step - accuracy: 0.9066 - loss: 0.3188 - val_accuracy: 0.9570 - val_loss: 0.1465
Epoch 2/10
430/430 - 2s - 4ms/step - accuracy: 0.9620 - loss: 0.1239 - val_accuracy: 0.9756 - val_loss: 0.0833
Epoch 3/10
430/430 - 2s - 5ms/step - accuracy: 0.9758 - loss: 0.0782 - val_accuracy: 0.9828 - val_loss: 0.0621
Epoch 4/10
430/430 - 2s - 4ms/step - accuracy: 0.9823 - loss: 0.0563 - val_accuracy: 0.9854 - val_loss: 0.0526
Epoch 5/10
430/430 - 2s - 4ms/step - accuracy: 0.9874 - loss: 0.0428 - val_accuracy: 0.9862 - val_loss: 0.0477
Epoch 6/10
430/430 - 2s - 4ms/step - accuracy: 0.9904 - loss: 0.0335 - val_accuracy: 0.9880 - val_loss: 0.0440
Epoch 7/10
430/430 - 2s - 4ms/step - accuracy: 0.9925 - loss: 0.0264 - val_accuracy: 0.9874 - val_loss: 0.0433
Epoch 8/10
430/430 - 2s - 4ms/step - accuracy: 0.9944 - loss: 0.0208 - val_accuracy: 0.9892 - val_loss: 0.0432
Epoch 9/10
430/430 - 2s - 4ms/step - accuracy: 0.9959 - loss: 0.0163 - val_accuracy: 0.9892 - val_loss: 0.0436


In [11]:
from tensorflow.keras.datasets import cifar10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 78s 0us/step


In [12]:
# 2) Normalizar para [0,1]
X_train = (X_train / 255.0).astype("float32")
X_test  = (X_test  / 255.0).astype("float32")

In [13]:
# 3) Separar validação (ex.: 5k amostras para val; 40k para treino)
VAL_SIZE = 5000
X_val, y_val = X_train[:VAL_SIZE], y_train[:VAL_SIZE]
X_tr,  y_tr  = X_train[VAL_SIZE:], y_train[VAL_SIZE:]

print("CIFAR-10 shapes:",
      "\n  X_tr:", X_tr.shape, "y_tr:", y_tr.shape,
      "\n  X_val:", X_val.shape, "y_val:", y_val.shape,
      "\n  X_test:", X_test.shape, "y_test:", y_test.shape)

CIFAR-10 shapes: 
  X_tr: (45000, 32, 32, 3) y_tr: (45000, 1) 
  X_val: (5000, 32, 32, 3) y_val: (5000, 1) 
  X_test: (10000, 32, 32, 3) y_test: (10000, 1)


In [14]:
# construir o LeNet-5 igual fizemos com a MNIST
cifar_model = build_lenet5(
    input_shape=(32, 32, 3),
    num_classes=10 # car, bird, cat, ...
)

# optimizers
    # default = SGD
    # outras opções mais modernas = Adam , AdamW

# compilar o modelo
cifar_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'] # além da loss, queremos monitorar a acurácia
)

print()
cifar_model.summary()
print()

Model: "LeNet5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 6)      │           456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_2             │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_3             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 1, 1, 120)      │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,006 (242.21 KB)

 Trainable params: 62,006 (242.21 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
#Treinamento do Modelo
history_cifar = cifar_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    verbose=2,
)

Epoch 1/30
391/391 - 8s - 20ms/step - accuracy: 0.3327 - loss: 1.8721 - val_accuracy: 0.3878 - val_loss: 1.7341
Epoch 2/30
391/391 - 2s - 4ms/step - accuracy: 0.3911 - loss: 1.7308 - val_accuracy: 0.4094 - val_loss: 1.6691
Epoch 3/30
391/391 - 2s - 4ms/step - accuracy: 0.4161 - loss: 1.6615 - val_accuracy: 0.4304 - val_loss: 1.6004
Epoch 4/30
391/391 - 2s - 4ms/step - accuracy: 0.4394 - loss: 1.5922 - val_accuracy: 0.4592 - val_loss: 1.5354
Epoch 5/30
391/391 - 2s - 6ms/step - accuracy: 0.4586 - loss: 1.5311 - val_accuracy: 0.4744 - val_loss: 1.4799
Epoch 6/30
391/391 - 3s - 7ms/step - accuracy: 0.4750 - loss: 1.4778 - val_accuracy: 0.4916 - val_loss: 1.4305
Epoch 7/30
391/391 - 3s - 7ms/step - accuracy: 0.4913 - loss: 1.4320 - val_accuracy: 0.5122 - val_loss: 1.3859
Epoch 8/30
391/391 - 2s - 5ms/step - accuracy: 0.5066 - loss: 1.3920 - val_accuracy: 0.5242 - val_loss: 1.3481
Epoch 9/30
391/391 - 2s - 6ms/step - accuracy: 0.5197 - loss: 1.3566 - val_accuracy: 0.5378 - val_loss: 1.3141


In [17]:
loss, acc = cifar_model.evaluate(X_test, y_test, verbose=0)

print(f"\n[TEST] Acc: {acc*100:.2f}%")


[TEST] Acc: 56.77%
